# Week 3 — Transfer learning for our PSA MT (EN/SW ↔ Ekegusii)

This notebook is our one-click runbook: GPU check → setup → zero-shot baselines →
fine-tuning the ablation matrix → demo → results table. Run cells top to bottom;
every cell is safe to re-run (finished runs are skipped automatically).

**First:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# 1) GPU check — we want a T4 or better before burning time on training.
import torch
print('torch:', torch.__version__)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print('GPU:', name)
    if 'T4' not in name and torch.cuda.get_device_capability(0) < (7, 5):
        print('WARNING: expected a T4-or-better GPU; training may be slow.')
else:
    print('WARNING: no GPU detected — switch the runtime to a GPU before continuing.')

## 2) Grab the team repo and install dependencies
Clones our GitHub repo and installs both requirement files (re-running just re-uses the existing clone).

In [ ]:
import os
if not os.path.exists('Machine_Translation_For_PSA_Group10'):
    !git clone https://github.com/LEVIN0/Machine_Translation_For_PSA_Group10
%cd Machine_Translation_For_PSA_Group10
!pip install -q -r requirements.txt -r requirements-training.txt

## 3) Weights & Biases login (optional)
We track runs on W&B. **No account / no key? Skip this cell** — every run still writes JSON logs to `runs/` and the results table in cell 9 works without W&B.

In [ ]:
import os
if os.environ.get('WANDB_API_KEY'):
    !wandb login $WANDB_API_KEY
    print('wandb: logged in via WANDB_API_KEY')
else:
    print('No WANDB_API_KEY set — using offline JSON-only logging.')
    print('To enable the dashboard: get a key at https://wandb.ai/authorize,')
    print('then run  !wandb login  in a new cell and paste it.')
    os.environ['WANDB_MODE'] = 'offline'

## 4) Build the held-out Ekegusii benchmark
**Finding (verified):** FLORES-200 contains **no Ekegusii** — the archive has 204 languages and `guz_Latn` is not among them — and NLLB-200's tokenizer has no `guz_Latn` token either. Neither Week 3 model has any Ekegusii pretraining, so there is no off-the-shelf Ekegusii benchmark. Our own held-out **test split** (138 eng–guz pairs from the lecturer gold data) is the benchmark: `guz_test.tsv` is **evaluation-only** (project rule — never trained on). Ekegusii *training* pairs come from the train split (the only guz source).

In [ ]:
!python scripts/build_guz_benchmark.py

## 5) Zero-shot baselines (`zs_mt5`, `zs_nllb`)
How good are the off-the-shelf models with **no training**? These are eval-only entries in the ablation matrix, so this cell is quick.

In [ ]:
import time
t0 = time.time()
!python scripts/run_ablations.py --matrix quick --only zs_mt5,zs_nllb --eval-n 200
print(f'zero-shot evals took {(time.time() - t0) / 60:.1f} min')

## 6) Fine-tuning runs (one cell per run, with timers)
Each cell fine-tunes one matrix entry and evaluates it on the dev + guz-benchmark specs. Expect ~20–30 min for mt5-small and ~35–50 min for NLLB-600M per run on a T4. Re-running a finished cell is a no-op, so you can split the matrix across Colab sessions (see docs/week3_colab_guide.md).

In [ ]:
# ft_nllb_base — NLLB fine-tuned on our EN↔SW PSA pairs — the workhorse run.
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_nllb_base --eval-n 200
print(f'ft_nllb_base took {(time.time() - t0) / 60:.1f} min')

In [ ]:
# ft_mt5_base — Same recipe on mT5-small — our second base model.
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_mt5_base --eval-n 200
print(f'ft_mt5_base took {(time.time() - t0) / 60:.1f} min')

In [ ]:
# ft_nllb_freeze — Low-resource trick: freeze the encoder, train the decoder only.
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_nllb_freeze --eval-n 200
print(f'ft_nllb_freeze took {(time.time() - t0) / 60:.1f} min')

In [ ]:
# ft_nllb_guz50 — Few-shot Ekegusii: capped at 50 PSA-sourced guz train pairs (direction=all).
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_nllb_guz50 --eval-n 200
print(f'ft_nllb_guz50 took {(time.time() - t0) / 60:.1f} min')

In [ ]:
# ft_nllb_guz200 — Few-shot Ekegusii with 200 PSA-sourced guz train pairs.
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_nllb_guz200 --eval-n 200
print(f'ft_nllb_guz200 took {(time.time() - t0) / 60:.1f} min')

In [ ]:
# ft_mt5_guz200 — Transfer comparison: the guz200 recipe on mT5 instead of NLLB.
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_mt5_guz200 --eval-n 200
print(f'ft_mt5_guz200 took {(time.time() - t0) / 60:.1f} min')

## 7) (Optional) Back-translation augmentation + `ft_nllb_aug`
We translate our English-only PSA rows to Swahili with the best fine-tuned checkpoint, then retrain with the synthetic pairs. Skip if short on time — the runner warns and skips `ft_nllb_aug` if the CSV is missing.

In [ ]:
from pathlib import Path
aug_csv = Path('data/processed/augmented_backtranslation.csv')
if not aug_csv.exists():
    from training.augment import backtranslate
    ckpt = Path('runs/ft_nllb_base/checkpoint-best')
    if not ckpt.exists():
        ckpt = Path('runs/ft_mt5_base/checkpoint-best')
    print('back-translating with', ckpt)
    backtranslate(ckpt, Path('data/processed/splits'), aug_csv)
import time
t0 = time.time()
!python scripts/run_ablations.py --only ft_nllb_aug --eval-n 200
print(f'ft_nllb_aug took {(time.time() - t0) / 60:.1f} min')

## 8) The demo — our success criterion 🎉
Translates our 8 demo PSAs into all applicable targets (EN↔SW and →Ekegusii) using the newest trained checkpoint, and prints a clean table.

In [ ]:
!python scripts/translate.py --demo

## 9) Results table
Scans `runs/` and writes `reports/week3_results.md` (missing values show as —).

In [ ]:
!python scripts/run_ablations.py --table-only
from IPython.display import Markdown, display
display(Markdown(open('reports/week3_results.md', encoding='utf-8').read()))

## 10) Download results / commit them back
Zip `runs/` + `reports/` for download (Colab wipes the disk when the runtime ends!). Optionally commit the results back to the repo from Colab.

In [ ]:
!zip -qr week3_results.zip runs reports
try:
    from google.colab import files
    files.download('week3_results.zip')
except ImportError:
    print('not on Colab — week3_results.zip is in the repo folder')

**Optional — commit back to GitHub from Colab** (needs a personal access token from github.com/settings/tokens). Uncomment and fill in:

```python
# import os
# os.environ['GH_TOKEN'] = 'ghp_...'
# !git config user.email 'you@example.com'
# !git config user.name 'Your Name'
# !git add runs reports
# !git commit -m 'Week 3: ablation results from Colab'
# !git push https://{os.environ['GH_TOKEN']}@github.com/LEVIN0/Machine_Translation_For_PSA_Group10 HEAD:week3-results
```

⚠️ Never paste the token into a notebook you share — clear the cell output after pushing.